**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/churchland-2012"], check=True)
    os.chdir("/content/encore/entries/churchland-2012")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings checked on dandiset 000070, version 0.261122.2046

This notebook belongs to the reconciliation of attempt churchland-2012-02 with attempt
churchland-2012-01 (docs/protocol.md, "Repeat attempts"). churchland-2012-01 was made on version
0.260528.0919 and recorded eight findings. churchland-2012-02 reports the same problem again for
finding 1 (the sorted isolations and the other datasets are not in the dandiset). Findings 2 to 8,
which it did not report, are checked here on version 0.261122.2046, the version churchland-2012-02
tested. Every file is streamed from the archive with the entry's helper module `reach.py`; only
metadata, trial tables, the first spikes of a few units and short stretches of hand position are
read. Each section opens with the earlier statement.

In [ ]:
import time
import numpy as np
import pandas as pd
import requests
import reach

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
started = time.time()
assets = reach.list_assets()
paths = [a["path"] for a in assets]
files = {p: reach.open_asset(p) for p in paths}
short = {p: p.split("/")[1].replace("_behavior+ecephys.nwb", "") for p in paths}
print(f"dandiset {reach.DANDISET} version {reach.VERSION}: {len(paths)} files")

The ten files at the pinned version: subject, date, units, spikes, trials and maze conditions.

In [ ]:
rows = []
for p, f in files.items():
    tr = f["intervals/trials"]
    cond = tr["maze_condition"][:] if "maze_condition" in tr else None
    rows.append({"file": short[p], "subject": f["general/subject/subject_id"][()].decode(),
                 "session_start_time": f["session_start_time"][()].decode(),
                 "units": len(f["units/spike_times_index"]), "spikes": len(f["units/spike_times"]),
                 "trials": len(tr["start_time"]),
                 "maze conditions": None if cond is None else int(len(np.unique(cond[cond > 0]))),
                 "hand samples": len(f["processing/behavior/Position/Hand/timestamps"])})
print(pd.DataFrame(rows).to_string(index=False))

## Earlier finding 2 (format_error)

> In nine of the ten files (all written by the first conversion) the first entry of
> units/spike_times_index is the true end of unit 0 modulo 65536. Unit 0 is cut short (in J-array
> 61030 of its 257638 spikes) and its remaining spikes open unit 1, whose times then jump from the
> end of the session back to its start.

For each file: the stored first entry of `units/spike_times_index`, the spikes of units 0 and 1 it
gives, whether each of the two trains is sorted, and their first and last spike against the span of
the trials. A wrapped index would leave unit 1 unsorted, starting near the end of the session.

In [ ]:
rows = []
for p, f in files.items():
    si = f["units/spike_times_index"][:].astype(np.int64)
    st = f["units/spike_times"][: si[1]]
    tr = f["intervals/trials"]
    t0, t1 = float(tr["start_time"][:].min()), float(tr["stop_time"][:].max())
    u0, u1 = st[: si[0]], st[si[0]: si[1]]
    rows.append({"file": short[p], "index[0]": int(si[0]), "index[0] < 65536": bool(si[0] < 65536),
                 "unit 0 spikes": len(u0), "unit 1 spikes": len(u1),
                 "unit 0 sorted": bool(np.all(np.diff(u0) >= 0)), "unit 1 sorted": bool(np.all(np.diff(u1) >= 0)),
                 "unit 0 first, last (s)": f"{u0[0]:.1f}, {u0[-1]:.1f}", "unit 1 first, last (s)": f"{u1[0]:.1f}, {u1[-1]:.1f}",
                 "trials span (s)": f"{t0:.1f}, {t1:.1f}"})
print(pd.DataFrame(rows).to_string(index=False))

## Earlier finding 3 (format_error)

> In five Nitschke files (ses-20090812, 20090819, 20090910, 20090920 and 20100923) the clock
> restarts at recording segments after the first, as the conversion offsets only one segment.
> Trials overlap in time (2094 overlapping pairs in N-array), and every unit's spike times go
> backwards at each restart, so spikes cannot be placed in trials by time; they can be placed by
> their storage order, which the notebook shows decodes reach direction in every segment.

For each file: how often a trial starts before the one before it, how many pairs of trials overlap
in time, the backward steps in the spike times of units 0, 95, 96 and 191 (the first and last unit
of each array), and the smallest share of those units' spikes that fall inside a trial interval.
The Nitschke files are now named with the year 2010 (see finding 8).

In [ ]:
rows = []
for p, f in files.items():
    tr = f["intervals/trials"]
    starts, stops = tr["start_time"][:], tr["stop_time"][:]
    o = np.argsort(starts)
    si = f["units/spike_times_index"][:].astype(np.int64)
    lo = np.r_[0, si[:-1]]
    back, inside = {}, []
    for u in (0, 95, 96, 191):
        s = f["units/spike_times"][lo[u]: si[u]]
        back[u] = int(np.sum(np.diff(s) < 0))
        k = np.searchsorted(starts[o], s, "right") - 1
        inside.append(np.mean((k >= 0) & (s <= stops[o][np.maximum(k, 0)] + 1e-6)))
    rows.append({"file": short[p], "trial starts going backwards": int(np.sum(np.diff(starts) < 0)),
                 "overlapping trial pairs": int(np.sum(starts[o][1:] < stops[o][:-1])),
                 "backward steps in units 0, 95, 96, 191": ", ".join(str(back[u]) for u in back),
                 "share of their spikes inside a trial": round(float(np.min(inside)), 4)})
print(pd.DataFrame(rows).to_string(index=False))

## Earlier finding 4 (metadata_error)

> Hand position is stated in metres (with conversion NaN); its values span about -220 to +165 on
> the scale of the target positions (up to 145) and the median speed peaks at 600 to 730 per second
> during a reach, so they are in mm.

For each file: the unit and conversion of `processing/behavior/Position/Hand/data`, the range of
the stored values over its first 300000 samples (about five minutes) against the hit target
positions, and the 99th percentile of the speed once the conversion is applied.

In [ ]:
rows = []
for p, f in files.items():
    h = f["processing/behavior/Position/Hand"]
    d = h["data"]
    n = min(300000, d.shape[0])
    x = d[:n]
    t = h["timestamps"][:n]
    tr = f["intervals/trials"]
    tp = tr["hit_target_position"][:]
    dt = np.diff(t)
    v = np.linalg.norm(np.diff(x, axis=0), axis=1)[np.abs(dt - 0.001) < 1e-6] / 0.001
    rows.append({"file": short[p], "unit": d.attrs["unit"], "conversion": float(d.attrs["conversion"]),
                 "stored x range": f"{np.nanmin(x[:, 0]):.1f} to {np.nanmax(x[:, 0]):.1f}",
                 "stored y range": f"{np.nanmin(x[:, 1]):.1f} to {np.nanmax(x[:, 1]):.1f}",
                 "hit targets (stored)": f"{np.nanmin(tp):.0f} to {np.nanmax(tp):.0f}",
                 "99th pct speed x conversion (m/s)": round(float(np.nanpercentile(v, 99) * d.attrs["conversion"]), 3)})
print(f"first {n} samples of each file")
print(pd.DataFrame(rows).to_string(index=False))

## Earlier finding 5 (metadata_insufficient)

> The hand position ends a median 35 mm (Jenkins) and 52 to 53 mm (Nitschke) below the reached
> target in y, while the cursor lands on the target; the offset between hand and screen
> coordinates is not described in the files.

The description of the trial table's `cursor_offset` column, and, for the selected trials of the
two array datasets, the hand position at `move_ends_time` minus the hit target, the cursor offset,
and the hand position against the target minus the offset.

In [ ]:
tr0 = files[reach.SESSIONS["J-array"]]["intervals/trials"]
print("cursor_offset description:", tr0["cursor_offset"].attrs.get("description"))
rows = []
for name in ("J-array", "N-array"):
    f = files[reach.SESSIONS[name]]
    d = reach.load_trials(f)
    sel = reach.select_trials(d) & d.move_ends_time.notna()
    d = d[sel]
    h = f["processing/behavior/Position/Hand"]
    ts = h["timestamps"][:]
    k = np.clip(np.searchsorted(ts, d.move_ends_time.values), 0, len(ts) - 1)
    lo, hi = int(k.min()), int(k.max()) + 1
    xy = h["data"][lo:hi][k - lo]
    hand = xy - d[["hit_x", "hit_y"]].values
    off = d[["cursor_dx", "cursor_dy"]].values
    rows.append({"session": name, "trials": len(d),
                 "sample at move end within (ms)": round(float(np.max(np.abs(ts[k] - d.move_ends_time.values)) * 1000), 1),
                 "hand minus target, median x, y (mm)": f"{np.median(hand[:, 0]):.1f}, {np.median(hand[:, 1]):.1f}",
                 "cursor_offset, median x, y (mm)": f"{np.median(off[:, 0]):.1f}, {np.median(off[:, 1]):.1f}",
                 "cursor_offset values": ", ".join(f"({a:g}, {b:g})" for a, b in sorted(set(map(tuple, off)))),
                 "hand minus (target minus offset), median x, y (mm)": f"{np.median(hand[:, 0] + off[:, 0]):.1f}, {np.median(hand[:, 1] + off[:, 1]):.1f}",
                 "median distance from target minus offset (mm)": round(float(np.median(np.linalg.norm(hand + off, axis=1))), 1)})
print(pd.DataFrame(rows).T.to_string(header=False))

## Earlier finding 6 (metadata_insufficient)

> The Units table carries only the default description ("Autogenerated by NWBFile") and does not
> say whether its units are sorted neurons, multi-unit activity or threshold crossings. Their
> number (one per electrode) and rates (8.8 to 300 spikes/s over trial time in the three sessions
> analysed) are consistent with channel-level activity.

The description and columns of the Units table in every file.

In [ ]:
rows = []
for p, f in files.items():
    u = f["units"]
    rows.append({"file": short[p], "units description": u.attrs.get("description"),
                 "columns": ", ".join(c for c in u.attrs["colnames"])})
print(pd.DataFrame(rows).to_string(index=False))

## Earlier finding 7 (metadata_insufficient)

> The dandiset does not say which of its ten sessions are the paper's datasets. Two match the
> J-array and N-array dates; the other eight (27-condition and two-condition maze sessions, and one
> 108-condition session) are not described.

The dandiset's description and the paper it is described by, and, in every file, the session
description, session id and keywords, searched for a name of the paper's datasets.

In [ ]:
API = "https://api.dandiarchive.org/api"
meta = requests.get(f"{API}/dandisets/{reach.DANDISET}/versions/{reach.VERSION}/", timeout=60).json()
print("Dandiset description:", meta["description"])
print("Described by:", [r.get("identifier") for r in meta.get("relatedResource", []) if r.get("relation") == "dcite:IsDescribedBy"])
print("Mentions of the paper's dataset names (J-array, N-array, 2012) in the dandiset metadata:",
      [w for w in ("J-array", "N-array", "Jarray", "Narray", "jPCA", "rotational") if w.lower() in str(meta).lower()])
rows = []
for p, f in files.items():
    g = f["general"]
    txt = lambda k: g[k][()].decode() if k in g and isinstance(g[k][()], bytes) else (str(g[k][()]) if k in g else "")
    allmeta = " ".join([f["session_description"][()].decode(), txt("experiment_description"), txt("session_id"),
                        txt("keywords"), txt("notes")]).lower()
    rows.append({"file": short[p], "session_description": f["session_description"][()].decode(),
                 "session_id": txt("session_id"), "keywords": txt("keywords")[:60],
                 "names a dataset of the paper": any(w in allmeta for w in ("j-array", "n-array", "jarray", "narray", "nature11129", "churchland"))})
print(pd.DataFrame(rows).to_string(index=False))
print("experiment_description of the first file:", files[paths[0]]["general/experiment_description"][()].decode())

## Earlier finding 8 (other, about the data)

> The first conversion's code sets the year of every session date to 2009. The file sub-Nitschke
> ses-20090910 has 108 conditions with the same reached target as N-array (2010) for all of them,
> and is consistent with the N-9-10-2010 dataset of Supplementary Figure 5; if so, its date is a
> year early. The other 2009 Nitschke dates may be affected in the same way.

The Nitschke file names at the earlier version and at this one, the session start dates in the
files, and the reached target of each condition in ses-20100910 against N-array (ses-20100923).

In [ ]:
earlier = requests.get(f"{API}/dandisets/{reach.DANDISET}/versions/0.260528.0919/assets/?page_size=200", timeout=60).json()["results"]
print("Nitschke files at version 0.260528.0919:", sorted(a["path"].split("/")[1].split("_")[1] for a in earlier if "Nitschke" in a["path"]))
print("Nitschke files at version", reach.VERSION, ":", sorted(short[p].split("_")[1] for p in paths if "Nitschke" in p))
def targets(f):
    d = reach.load_trials(f)
    ok = (d.task_success == 1) & (d.maze_condition > 0)
    return d[ok].groupby("maze_condition")[["hit_x", "hit_y"]].agg(lambda v: v.mode().iloc[0])
a = targets(files[reach.SESSIONS["N-9-10-2010"]])
b = targets(files[reach.SESSIONS["N-array"]])
common = a.index.intersection(b.index)
print(f"ses-20100910 against ses-20100923: {len(common)} conditions in both; same reached target in "
      f"{(a.loc[common] == b.loc[common]).all(axis=1).mean():.2f}")
print({short[p]: files[p]["session_start_time"][()].decode()[:10] for p in paths})
print(f"\nRun time {(time.time() - started) / 60:.1f} min")

The output of each section shows whether the earlier problem is still there. The conclusion for
each finding is in the reconciliation block of `attempts/churchland-2012-02.yaml`.